# 05: DINOv2 + Supervised Contrastive Learning
## RBC 9종 클래스 분리 (Foundation Model 기반)

- **목표**: 이미지 생성 X → latent space에서 9종 클래스 분리 (SCI 논문)
- **방법**: DINOv2 frozen feature → MLP projection head → SupCon Loss
- **vs 04 차이**:
  - cVAE-GAN 폐기 (이미지 생성 불필요, D 압도, KL 폭발)
  - DINOv2: 1억장 사전학습 → 431개 소수 클래스도 학습 가능
  - ClassBalancedBatchSampler: 매 배치에 9클래스 균등 포함
  - SupCon Loss: 버그 없는 올바른 구현 (항상 양수 보장)

### 파이프라인
```
이미지 (96×96)
  → DINOv2 ViT-S/14 (frozen, 1억장 사전학습)
  → 384차원 feature
  → MLP Projection Head (384→512→512→128)
  → SupCon Loss (같은 클래스 가깝게, 다른 클래스 멀게)
  → t-SNE / Silhouette Score로 분리도 평가
```

In [ ]:
import os
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from collections import defaultdict
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, Sampler
from torchvision import transforms
from PIL import Image

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {DEVICE}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
# ── 설정 ──────────────────────────────────────────────
BASE_DIR   = "/home/smile/work/a_CKM_mo/python39/Claude/"
CROP_DIR   = os.path.join(BASE_DIR, "crops")
MODEL_DIR  = os.path.join(BASE_DIR, "models/dinov2_supcon")
FEAT_DIR   = os.path.join(MODEL_DIR, "features")     # DINOv2 feature 캐시
CKPT_DIR   = os.path.join(MODEL_DIR, "checkpoints")
SAMPLE_DIR = os.path.join(MODEL_DIR, "samples")

for d in [MODEL_DIR, FEAT_DIR, CKPT_DIR, SAMPLE_DIR]:
    os.makedirs(d, exist_ok=True)

CLASS_NAMES = [
    'RBC', 'Reticulocyte', 'Echinocyte', 'Elliptocyte',
    'Schistocyte', 'Stomatocyte', 'TargetCell', 'TearDropCell', 'Nucleated'
]
N_CLASSES = len(CLASS_NAMES)
CLASS2ID  = {c: i for i, c in enumerate(CLASS_NAMES)}

# ── DINOv2 설정 ──
DINO_MODEL    = 'dinov2_vits14'   # ViT-S/14 → 384차원 (가볍고 검증됨)
DINO_FEAT_DIM = 384
IMG_SIZE      = 98                # 14의 배수 (96→98=14×7): DINOv2 패치 크기 맞춤

# ── Projection Head ──
PROJ_DIM   = 128
HIDDEN_DIM = 512

# ── 학습 ──
N_SAMPLES_PER_CLASS  = 16         # 배치당 클래스별 샘플 수
BATCH_SIZE           = N_SAMPLES_PER_CLASS * N_CLASSES  # 144
N_BATCHES_PER_EPOCH  = 200        # epoch당 배치 수
EPOCHS               = 150
LR                   = 1e-3
WEIGHT_DECAY         = 1e-4
TEMPERATURE          = 0.07       # SupCon temperature (낮을수록 클래스 경계 선명)
SAVE_EVERY           = 10

print(f"클래스: {CLASS_NAMES}")
print(f"DINOv2 feature dim: {DINO_FEAT_DIM}")
print(f"Batch: {BATCH_SIZE} = {N_SAMPLES_PER_CLASS} per class × {N_CLASSES} classes")
print(f"Epoch당 {N_BATCHES_PER_EPOCH} 배치 × {EPOCHS} epochs")

In [ ]:
# ── DINOv2 로드 ───────────────────────────────────────
dinov2 = torch.hub.load('facebookresearch/dinov2', DINO_MODEL)
dinov2 = dinov2.to(DEVICE).eval()
for p in dinov2.parameters():
    p.requires_grad = False

print(f"DINOv2 ({DINO_MODEL}) 로드 완료")
print(f"파라미터: {sum(p.numel() for p in dinov2.parameters()):,} (전부 frozen)")

In [ ]:
# ── Feature 추출 & 캐시 ────────────────────────────────
# DINOv2는 ImageNet normalization 사용
dino_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])


def extract_and_cache(split):
    """DINOv2로 전체 이미지 feature 추출 후 캐시 저장 (재실행 시 캐시 재사용)"""
    feat_path = os.path.join(FEAT_DIR, f'{split}_features.npy')
    cls_path  = os.path.join(FEAT_DIR, f'{split}_labels.npy')

    if os.path.exists(feat_path):
        print(f"[{split}] 캐시 로드: {feat_path}")
        return np.load(feat_path), np.load(cls_path)

    all_feats, all_labels = [], []

    for cls_name in CLASS_NAMES:
        cls_dir = Path(CROP_DIR) / split / cls_name
        if not cls_dir.exists():
            print(f"  [경고] {cls_dir} 없음")
            continue
        cls_id = CLASS2ID[cls_name]
        files  = sorted(cls_dir.glob('*.jpg'))

        batch_imgs = []
        for p in tqdm(files, desc=f'  {cls_name:15s}'):
            img = Image.open(p).convert('RGB')
            batch_imgs.append(dino_transform(img))

            if len(batch_imgs) == 256:
                with torch.no_grad():
                    feats = dinov2(torch.stack(batch_imgs).to(DEVICE)).cpu().numpy()
                all_feats.append(feats)
                all_labels.extend([cls_id] * len(batch_imgs))
                batch_imgs = []

        if batch_imgs:
            with torch.no_grad():
                feats = dinov2(torch.stack(batch_imgs).to(DEVICE)).cpu().numpy()
            all_feats.append(feats)
            all_labels.extend([cls_id] * len(batch_imgs))

        print(f"  {cls_name:15s}: {len(files):,}개")

    all_feats  = np.concatenate(all_feats)
    all_labels = np.array(all_labels)

    np.save(feat_path, all_feats)
    np.save(cls_path,  all_labels)
    print(f"[{split}] 완료: {all_feats.shape} → {feat_path}")
    return all_feats, all_labels


print("=== Train ===")
train_feats, train_labels = extract_and_cache('train')
print("=== Val ===")
val_feats, val_labels = extract_and_cache('val')

# 클래스별 분포 확인
print("\n클래스별 Train 분포:")
for cls_id, cls_name in enumerate(CLASS_NAMES):
    n = (train_labels == cls_id).sum()
    print(f"  {cls_name:15s}: {n:,}")

In [ ]:
# ── Dataset & Class-Balanced Sampler ──────────────────

class FeatureDataset(Dataset):
    """캐시된 DINOv2 feature로 구성한 Dataset (이미지 로드 불필요)"""
    def __init__(self, features, labels):
        self.features = torch.tensor(features, dtype=torch.float32)
        self.labels   = torch.tensor(labels,   dtype=torch.long)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return self.features[idx], self.labels[idx]


class ClassBalancedBatchSampler(Sampler):
    """
    매 배치마다 모든 클래스에서 n_samples_per_class개씩 균등 샘플링.
    SupCon Loss에 필수: 각 배치에 positive pair가 반드시 존재해야 함.
    소수 클래스(431개)도 매 배치에 포함 (WeightedSampler는 같은 이미지 반복만 함).
    """
    def __init__(self, labels, n_classes, n_samples_per_class, n_batches):
        self.n_classes            = n_classes
        self.n_samples_per_class  = n_samples_per_class
        self.n_batches            = n_batches

        self.class_indices = defaultdict(list)
        for idx, lbl in enumerate(labels):
            self.class_indices[int(lbl)].append(idx)

    def __iter__(self):
        for _ in range(self.n_batches):
            batch = []
            for cls_id in range(self.n_classes):
                indices = self.class_indices[cls_id]
                k = self.n_samples_per_class
                # 소수 클래스는 replacement=True로 보완
                sampled = random.choices(indices, k=k) if len(indices) < k \
                          else random.sample(indices, k)
                batch.extend(sampled)
            random.shuffle(batch)
            yield batch

    def __len__(self):
        return self.n_batches


train_dataset = FeatureDataset(train_feats, train_labels)
val_dataset   = FeatureDataset(val_feats,   val_labels)

train_sampler = ClassBalancedBatchSampler(
    train_labels, N_CLASSES, N_SAMPLES_PER_CLASS, N_BATCHES_PER_EPOCH
)
train_loader = DataLoader(train_dataset, batch_sampler=train_sampler, num_workers=0)
val_loader   = DataLoader(val_dataset,   batch_size=512, shuffle=False, num_workers=0)

print(f"Train: {len(train_dataset):,}개  |  Val: {len(val_dataset):,}개")
print(f"배치: {BATCH_SIZE}개 ({N_SAMPLES_PER_CLASS}×{N_CLASSES}), epoch당 {N_BATCHES_PER_EPOCH} 배치")

In [ ]:
# ── 모델 & Loss 정의 ─────────────────────────────────

class ProjectionHead(nn.Module):
    """
    DINOv2 feature (384) → projection space (128)
    SupCon 논문 권장: 2-layer MLP + BN + ReLU
    """
    def __init__(self, in_dim=DINO_FEAT_DIM, hidden_dim=HIDDEN_DIM, out_dim=PROJ_DIM):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            nn.ReLU(inplace=True),
            nn.Linear(hidden_dim, hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            nn.ReLU(inplace=True),
            nn.Linear(hidden_dim, out_dim),
        )

    def forward(self, x):
        return F.normalize(self.net(x), dim=1)  # L2 정규화 → 코사인 공간


class SupConLoss(nn.Module):
    """
    Supervised Contrastive Loss (Khosla et al., NeurIPS 2020)
    - 항상 양수 보장 (04의 음수값 버그 수정)
    - 수치 안정성: max subtraction + 1e-8
    - positive pair 없는 anchor 자동 제외
    """
    def __init__(self, temperature=0.07):
        super().__init__()
        self.temperature = temperature

    def forward(self, features, labels):
        # features: [B, D] — 이미 L2 normalized
        # labels:   [B]
        device = features.device
        B = features.size(0)

        # 유사도 행렬 [B, B]
        sim = torch.matmul(features, features.T) / self.temperature

        # 수치 안정성: 각 행의 max 빼기
        sim_max, _ = sim.max(dim=1, keepdim=True)
        sim = sim - sim_max.detach()

        # Mask 생성
        mask_self = torch.eye(B, dtype=torch.bool, device=device)
        lbl_eq    = labels.unsqueeze(0) == labels.unsqueeze(1)     # [B, B]
        mask_pos  = lbl_eq & ~mask_self                            # 같은 클래스, self 제외

        # exp 행렬 (self 제외: denominator에서 자기 자신 제거)
        exp_sim = torch.exp(sim) * (~mask_self).float()            # [B, B]

        # log P = sim_pos - log(sum_all_except_self exp)
        log_prob = sim - torch.log(exp_sim.sum(dim=1, keepdim=True) + 1e-8)  # [B, B]

        # positive pair가 있는 anchor만 사용
        n_pos = mask_pos.float().sum(dim=1)                        # [B]
        valid = n_pos > 0

        if valid.sum() == 0:
            return torch.tensor(0.0, device=device, requires_grad=True)

        # positive pair의 log_prob 평균 (항상 양수)
        loss = -(mask_pos.float() * log_prob).sum(dim=1)           # [B]
        loss = (loss[valid] / n_pos[valid]).mean()
        return loss


model     = ProjectionHead().to(DEVICE)
criterion = SupConLoss(temperature=TEMPERATURE)

optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, EPOCHS, eta_min=1e-6)

print(f"ProjectionHead params: {sum(p.numel() for p in model.parameters()):,}")
print(f"  {DINO_FEAT_DIM} → {HIDDEN_DIM} → {HIDDEN_DIM} → {PROJ_DIM} (L2 norm)")

In [ ]:
# ── 학습 루프 ─────────────────────────────────────────
history = {'train_loss': [], 'val_loss': [], 'lr': []}
best_loss  = float('inf')
best_epoch = 0

for epoch in range(1, EPOCHS + 1):
    # ── Train ──
    model.train()
    train_loss = 0.0
    for feats, labels in train_loader:
        feats, labels = feats.to(DEVICE), labels.to(DEVICE)
        proj  = model(feats)
        loss  = criterion(proj, labels)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        train_loss += loss.item()
    train_loss /= len(train_loader)

    # ── Val ──
    model.eval()
    val_loss = 0.0
    with torch.no_grad():
        for feats, labels in val_loader:
            feats, labels = feats.to(DEVICE), labels.to(DEVICE)
            proj = model(feats)
            val_loss += criterion(proj, labels).item()
    val_loss /= len(val_loader)

    scheduler.step()
    cur_lr = scheduler.get_last_lr()[0]

    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)
    history['lr'].append(cur_lr)

    # Best model 저장
    if val_loss < best_loss:
        best_loss  = val_loss
        best_epoch = epoch
        torch.save({'epoch': epoch, 'model': model.state_dict(),
                    'history': history},
                   os.path.join(CKPT_DIR, 'best.pt'))

    if epoch % 10 == 0 or epoch == 1:
        print(f"[{epoch:03d}/{EPOCHS}] "
              f"Train:{train_loss:.4f}  Val:{val_loss:.4f}  "
              f"LR:{cur_lr:.2e}  [best={best_loss:.4f} @ep{best_epoch}]")

    if epoch % SAVE_EVERY == 0:
        torch.save({'epoch': epoch, 'model': model.state_dict(), 'history': history},
                   os.path.join(CKPT_DIR, f'ckpt_{epoch:04d}.pt'))

print(f"\n학습 완료 — best val loss: {best_loss:.4f} @ epoch {best_epoch}")

In [ ]:
# ── 학습 곡선 ─────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(history['train_loss'], label='Train')
axes[0].plot(history['val_loss'],   label='Val')
axes[0].set_title('SupCon Loss'); axes[0].set_xlabel('epoch')
axes[0].legend(); axes[0].grid(True)

axes[1].plot(history['lr'])
axes[1].set_title('Learning Rate (CosineAnnealing)')
axes[1].set_xlabel('epoch'); axes[1].grid(True)

plt.tight_layout()
plt.savefig(os.path.join(MODEL_DIR, 'loss_curve.png'), dpi=150)
plt.show()

In [ ]:
# ── t-SNE: Before vs After ────────────────────────────
from sklearn.manifold import TSNE

# Best 모델 로드
ckpt = torch.load(os.path.join(CKPT_DIR, 'best.pt'), weights_only=False)
model.load_state_dict(ckpt['model'])
model.eval()
print(f"Best epoch {ckpt['epoch']} 로드")


def subsample_balanced(feats, labels, max_per_class=500):
    """t-SNE 속도를 위해 클래스별 최대 max_per_class개로 서브샘플"""
    sel_feats, sel_labels = [], []
    rng = np.random.default_rng(SEED)
    for cls_id in range(N_CLASSES):
        idx = np.where(labels == cls_id)[0]
        idx = rng.choice(idx, min(max_per_class, len(idx)), replace=False)
        sel_feats.append(feats[idx])
        sel_labels.append(labels[idx])
    return np.concatenate(sel_feats), np.concatenate(sel_labels)


sub_feats, sub_labels = subsample_balanced(val_feats, val_labels, max_per_class=500)

# Projection (After)
with torch.no_grad():
    sub_proj = model(
        torch.tensor(sub_feats, dtype=torch.float32).to(DEVICE)
    ).cpu().numpy()

# sklearn 버전 호환: n_iter(구버전) → max_iter(1.2+)
import sklearn
sk_ver = tuple(int(x) for x in sklearn.__version__.split('.')[:2])
tsne_kwargs = dict(n_components=2, perplexity=40, random_state=SEED, n_jobs=-1)
if sk_ver >= (1, 2):
    tsne_kwargs['max_iter'] = 1000
else:
    tsne_kwargs['n_iter'] = 1000

print("t-SNE 계산 중 (Before: DINOv2 raw)...")
emb_before = TSNE(**tsne_kwargs).fit_transform(sub_feats)

print("t-SNE 계산 중 (After: SupCon projection)...")
emb_after  = TSNE(**tsne_kwargs).fit_transform(sub_proj)

colors = plt.cm.tab10(np.linspace(0, 1, N_CLASSES))

fig, axes = plt.subplots(1, 2, figsize=(22, 9))
for cls_id, cls_name in enumerate(CLASS_NAMES):
    mask = sub_labels == cls_id
    for ax, emb in zip(axes, [emb_before, emb_after]):
        ax.scatter(emb[mask, 0], emb[mask, 1],
                   c=[colors[cls_id]], label=cls_name, alpha=0.5, s=14)

axes[0].set_title('Before — DINOv2 raw feature (384dim)', fontsize=13)
axes[1].set_title(f'After  — SupCon projection (128dim, ep{ckpt["epoch"]})', fontsize=13)
for ax in axes:
    ax.legend(loc='best', fontsize=8, markerscale=3)
    ax.axis('off')

plt.suptitle('RBC 9종 클래스 분리: DINOv2 raw vs SupCon', fontsize=14)
plt.tight_layout()
plt.savefig(os.path.join(MODEL_DIR, 'tsne_before_after.png'), dpi=200)
plt.show()

In [ ]:
# ── 정량 지표 ─────────────────────────────────────────
from sklearn.metrics import silhouette_score
from scipy.spatial.distance import cdist


def pairwise_separation(features, labels, n_sample=300):
    """모든 클래스 쌍에 대해 Between/Within cosine distance 비율 계산"""
    rng = np.random.default_rng(SEED)
    results = {}
    for i in range(N_CLASSES):
        for j in range(i + 1, N_CLASSES):
            fi = features[labels == i]
            fj = features[labels == j]
            if len(fi) == 0 or len(fj) == 0:
                continue
            fi = fi[rng.choice(len(fi), min(n_sample, len(fi)), replace=False)]
            fj = fj[rng.choice(len(fj), min(n_sample, len(fj)), replace=False)]

            wi = cdist(fi, fi, 'cosine')[np.triu_indices(len(fi), k=1)].mean()
            wj = cdist(fj, fj, 'cosine')[np.triu_indices(len(fj), k=1)].mean()
            cross = cdist(fi, fj, 'cosine').ravel().mean()
            results[(CLASS_NAMES[i], CLASS_NAMES[j])] = cross / ((wi + wj) / 2 + 1e-8)
    return results


print("=" * 60)
print("정량 지표 (val set, 클래스별 최대 500개)")
print("=" * 60)

for tag, feats in [('Before (DINOv2 raw)', sub_feats),
                   ('After  (SupCon)     ', sub_proj)]:
    sil = silhouette_score(feats, sub_labels, metric='cosine',
                           sample_size=min(5000, len(sub_labels)), random_state=SEED)
    sep = pairwise_separation(feats, sub_labels)

    hard_pairs = [
        ('RBC', 'Reticulocyte'), ('RBC', 'Echinocyte'),
        ('Echinocyte', 'Elliptocyte'), ('Schistocyte', 'Stomatocyte'),
        ('TearDropCell', 'Echinocyte'),
    ]

    print(f"\n[{tag}]")
    print(f"  Silhouette Score: {sil:.4f}  (↑높을수록 분리 잘됨, 최대 1.0)")
    print(f"  분리도 (Between/Within, ↑높을수록 좋음):")
    for pair in hard_pairs:
        if pair in sep:
            print(f"    {pair[0]:15s} vs {pair[1]:15s}: {sep[pair]:.3f}")

# ── 분리도 히트맵 ──
sep_before = pairwise_separation(sub_feats, sub_labels)
sep_after  = pairwise_separation(sub_proj,  sub_labels)

n = N_CLASSES
mat_before = np.zeros((n, n))
mat_after  = np.zeros((n, n))
for i in range(n):
    for j in range(i + 1, n):
        pair = (CLASS_NAMES[i], CLASS_NAMES[j])
        mat_before[i, j] = mat_before[j, i] = sep_before.get(pair, 0)
        mat_after[i, j]  = mat_after[j, i]  = sep_after.get(pair, 0)

fig, axes = plt.subplots(1, 2, figsize=(18, 7))
for ax, mat, title in [(axes[0], mat_before, 'Before (DINOv2 raw)'),
                       (axes[1], mat_after,  'After (SupCon)')]:
    im = ax.imshow(mat, cmap='RdYlGn', vmin=0.5, vmax=2.0)
    ax.set_xticks(range(n))
    ax.set_xticklabels(CLASS_NAMES, rotation=45, ha='right', fontsize=8)
    ax.set_yticks(range(n))
    ax.set_yticklabels(CLASS_NAMES, fontsize=8)
    ax.set_title(f'분리도 (Between/Within cosine) — {title}', fontsize=11)
    plt.colorbar(im, ax=ax)

plt.tight_layout()
plt.savefig(os.path.join(MODEL_DIR, 'separation_heatmap.png'), dpi=150)
plt.show()

In [ ]:
# ── 최종 모델 저장 ────────────────────────────────────
torch.save({
    'model': model.state_dict(),
    'history': history,
    'config': {
        'dino_model':    DINO_MODEL,
        'dino_feat_dim': DINO_FEAT_DIM,
        'proj_dim':      PROJ_DIM,
        'hidden_dim':    HIDDEN_DIM,
        'temperature':   TEMPERATURE,
        'n_classes':     N_CLASSES,
        'class_names':   CLASS_NAMES,
        'best_epoch':    best_epoch,
        'best_val_loss': best_loss,
    }
}, os.path.join(MODEL_DIR, 'dinov2_supcon_final.pt'))
print(f"저장 완료: {MODEL_DIR}/dinov2_supcon_final.pt")
print(f"Best val loss: {best_loss:.4f} @ epoch {best_epoch}")